# TNG-SN Cloud Mode Demo

This notebook shows how to run across multiple snapshots on the IllustrisTNG JupyterLab server.

- Assumes snapshots are locally mounted on the server.
- Uses the new `backend.mode = 'cloud'` which avoids persisting temp data.
- Optionally enables `cleanup_temp` to remove transient artifacts after runs.

## 1. Configure Execution Mode (Local vs Cloud) and Paths

We configure cloud mode by setting backend options. In cloud mode, we avoid persisting temporary data (ages, caches) and optionally enable cleanup.

We also derive paths from environment variables (e.g., `CLOUD_MODE`, `SNAPSHOT_ROOT`).

In [ ]:
import os, json, pathlib, shutil, getpass, logging
from snsims.simulation import TNGSNSimulation

CLOUD_MODE = os.environ.get('CLOUD_MODE', '1') == '1'
SNAPSHOT_ROOT = os.environ.get('SNAPSHOT_ROOT', '/home/jovyan/data/IllustrisTNG')
OUTPUT_DIR = os.environ.get('OUTPUT_DIR', 'simout_cloud')

# Configure backend for cloud mode
backend_cfg = {
    'mode': 'cloud' if CLOUD_MODE else 'local',
    'cleanup_temp': True,
    'persist_ages': False,
    'api_cache_dir': '.cache/tngsn_cloud'
}

# Validate paths
root = pathlib.Path.cwd()
snaproot = pathlib.Path(SNAPSHOT_ROOT)
assert snaproot.exists(), f"SNAPSHOT_ROOT missing: {snaproot}"
(root / OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('Cloud mode:', backend_cfg['mode']=='cloud')
print('Snapshot root:', str(snaproot))

## 2. Discover and Select Snapshots

Scan local snapshot directories/files and select which snapshots to process.

In [ ]:
import re, glob
from pathlib import Path

def find_snapshots(snaproot: Path, sim: str):
    # Example layout: /.../TNG100-1/output/ or similar; adapt as needed
    candidates = []
    for p in snaproot.rglob('snapdir_*'):
        m = re.search(r'snapdir_(\d+)', p.name)
        if m:
            candidates.append(int(m.group(1)))
    return sorted(set(candidates))

SIMULATION = os.environ.get('SIMULATION', 'TNG100-1')
SNAPSHOTS = find_snapshots(snaproot, SIMULATION)
print('Discovered snapshots:', SNAPSHOTS[:10], '... total', len(SNAPSHOTS))

# Filters
SNAP_MIN = int(os.environ.get('SNAP_MIN', SNAPSHOTS[0] if SNAPSHOTS else 0))
SNAP_MAX = int(os.environ.get('SNAP_MAX', SNAPSHOTS[-1] if SNAPSHOTS else 0))
SNAP_STEP = int(os.environ.get('SNAP_STEP', 1))
SELECTED = [s for s in SNAPSHOTS if SNAP_MIN <= s <= SNAP_MAX][::SNAP_STEP]
print('Selected snapshots:', SELECTED)

## 7. Batch Processor Over Snapshots with Progress Bar

We configure TNGSNSimulation for cloud mode and iterate snapshots.

In [ ]:
from tqdm import tqdm
import pandas as pd

cfg_overrides = {
    'paths__root_path': str(root),
    'simulation__name': SIMULATION,
    'simulation__output_dir': OUTPUT_DIR,
    'backend__mode': backend_cfg['mode'],
    'backend__cleanup_temp': backend_cfg['cleanup_temp'],
    'backend__persist_ages': backend_cfg['persist_ages'],
}

sim = TNGSNSimulation(None, **cfg_overrides)
results_by_snap = sim.run_snapshots(SELECTED, n_subhalos=int(os.environ.get('N_SUBHALOS', 2)))
print({k: (len(v) if isinstance(v, pd.DataFrame) else 0) for k,v in results_by_snap.items()})

## 18. Final Teardown and Temp Data Cleanup

If `backend.cleanup_temp` is True, temp data is removed automatically after each run. You can also call the cleanup explicitly if needed.